# Semaine 7 — Jour 2 : Orchestration et état durable

Objectif : Décrire les transitions valides d’un job métier.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Pour une étude complète, prévoir 2 à 3 heures avec entretien, challenge et revue. Le premier lancement Docker peut demander un temps supplémentaire.

Prérequis : Python 3.12 recommandé, HTTP/JSON, notions d’agents et de persistance des semaines précédentes. Les commandes Python utilisent uniquement la bibliothèque standard. n8n exige Docker Compose v2 pour le lab d’intégration.

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md) avant d’ouvrir [les corrigés](corriges/exercises_solution.md). Préparer [l’entretien](interview.md) et [le challenge](challenge.md).

Notebooks depuis la racine : `notebooks/week07/S7_J2_orchestration_etat.ipynb` et `notebooks/week07/teacher/S7_J2_orchestration_etat_teacher.ipynb`.

Critère de réussite : Reprendre un job après redémarrage sans confondre état et historique technique.

Le lab ne requiert aucune clé LLM. Ses réponses sont simulées et ne mesurent pas la qualité d’un modèle réel.


# Objectifs mesurables

- Décrire les transitions valides d’un job métier.
- Distinguer orchestration explicite et autonomie d’un agent.
- Reprendre un job après redémarrage sans confondre état et historique technique.

## Preuves attendues

Une sortie de lab reproductible, une explication de l’invariant étudié et une limite correctement identifiée. Le challenge approfondit le parcours essentiel sans être nécessaire pour finir la session de 60 minutes.


# Semaine 7 — Jour 2 : Orchestration et état durable


## 1. Du graphe au protocole métier
Un graphe indique l’ordre des appels, mais ne suffit pas à définir le comportement après une panne. Le job est-il déjà enregistré ? Le brouillon a-t-il été validé ? La réponse a-t-elle été publiée ? Une machine à états rend ces questions explicites. Dans le fil rouge, `received` indique l’enregistrement, `awaiting_review` indique un brouillon disponible et `published` signifie une publication locale simulée.

Deux branches terminent le processus sans publication : `rejected` et `expired`. Les erreurs utilisent `retry` ou `dead_letter`. Chaque état a un sens observable. Une règle simple doit rester vraie : aucun contenu ne figure dans `publications` avant une approbation valable. Une réponse de modèle ne peut jamais se substituer à cette approbation.

## 2. Qui orchestre ?
n8n déclenche `POST /tick` périodiquement. Le backend recherche les jobs éligibles et applique une étape. Le moteur possède la machine à états et le stockage; n8n possède les déclencheurs et le transport. Cette répartition évite un couplage à l’identifiant d’exécution n8n. Une exécution du scheduler peut traiter plusieurs jobs et un job peut nécessiter plusieurs exécutions.

L’orchestration explicite fixe le parcours avant l’exécution. Un agent peut choisir un outil ou adapter un plan à un résultat intermédiaire. Les semaines précédentes ont traité la boucle agentique; cette semaine étudie la maîtrise du processus autour de cette boucle. Il n’est pas nécessaire de refaire un mini-framework : notre module isolé démontre les invariants d’intégration.

## 3. Modéliser les transitions
L’ingestion ne prépare pas le brouillon. `process` ne traite que `received` ou `retry` si la date de reprise est atteinte. Appeler `process` sur un job en attente de validation ne génère pas un nouveau texte. `decide` n’accepte qu’un état en attente, un hash de proposition courant et une décision reconnue. Les répétitions d’une décision déjà appliquée sont idempotentes si elles sont identiques.

Une transition terminale ne revient pas silencieusement en arrière. Le replay est explicite et limité aux dead letters. Si le contrat ou le brouillon change, il faut une nouvelle version du processus ou une nouvelle demande; une approbation ancienne ne doit pas autoriser un texte modifié.

## 4. Persister avant de répondre
La transaction d’ingestion écrit le job et son audit avant de renvoyer l’accusé de réception. Un redémarrage retrouve donc les jobs acceptés. La connexion SQLite utilise ici une exécution séquentielle; elle illustre la durabilité d’un processus unique. Elle ne constitue pas un worker distribué : plusieurs processus nécessiteraient claim atomique, lease, fencing et tests de concurrence.

La génération se déroule avant la transaction qui stocke son résultat. Si le processus s’arrête entre ces deux étapes, le générateur peut être rappelé. Pour une lecture ou un LLM sans effet métier, le principal risque est un coût supplémentaire. Pour un appel qui écrit dans un système tiers, cette fenêtre change le problème et exige une stratégie d’idempotence adaptée.

## 5. Observer l’état
GET /jobs/{event_id} retourne l’état métier et le brouillon courant. GET /metrics expose un petit comptage JSON; ce n’est pas un endpoint Prometheus. La table audit enregistre des actions sans recopier le texte client. Un journal technique indique les appels; l’état métier permet de répondre à la question « où en est la demande ? ».

## À retenir
La reprise ne consiste pas à relancer aveuglément le graphe entier. Elle consiste à retrouver un état durable puis à appliquer uniquement une transition encore autorisée.


## Diagramme du jour

```mermaid
stateDiagram-v2
  [*] --> received
  received --> awaiting_review: proposition valide
  received --> retry: panne temporaire
  retry --> awaiting_review: reprise
  retry --> dead_letter: limite atteinte
  awaiting_review --> published: approbation
  awaiting_review --> rejected: refus
  awaiting_review --> expired: délai dépassé
```

Si votre lecteur Jupyter ne rend pas Mermaid, ouvrir le fichier .mmd avec un lecteur compatible.

# Exemple exécutable autonome

Ce fragment illustre le concept du jour. Le moteur complet et ses scénarios sont dans examples/week07.



In [ ]:
transitions = {"received": {"awaiting_review", "retry", "dead_letter"},
               "awaiting_review": {"published", "rejected", "expired"}}
def move(current, target):
    if target not in transitions.get(current, set()):
        raise ValueError("Transition interdite")
    return target
assert move("received", "awaiting_review") == "awaiting_review"
try:
    move("received", "published")
except ValueError:
    print("Publication directe bloquée")
else:
    raise AssertionError("Transition dangereuse")


# Lab J2 — Orchestration et état durable

Depuis la racine du dépôt :

```bash
python3 book/week07/day02/labs/run.py
```

Résultat attendu : **status=awaiting_review et une proposition valide**.

Le script exécute le scénario du jour contre une base en mémoire et ferme la connexion. Il ne crée aucun message externe. Inspecter `examples/week07/demo.py` et la méthode correspondante dans `workflow.py` pour relier la sortie à la transition étudiée.

## Travail à réaliser

1. Exécuter le scénario et expliquer chaque champ retourné.
2. Réaliser l’exercice 2 du jour avec une entrée différente.
3. Justifier l’absence de publication non autorisée.

Pour le parcours n8n, suivre [le guide d’installation](../../../../docs/week07/QUICKSTART.md). J1 configure la réception; J2 ajoute le scheduler; J3 teste les doublons; J4 observe les reprises et erreurs; J5 étudie le port de génération; J6 importe le workflow de décision; J7 exécute la checklist d’intégration.

Le scénario J4 utilise une horloge virtuelle. Les tests de reprise sont rapides et n’attendent pas réellement les délais. Le projet HTTP utilise une base fichier pour démontrer la persistance après redémarrage.


# Exercices

Faire une première tentative sans consulter les corrigés.

## Exercice 1 — Écrire la machine à états

Lister les sorties possibles de received, retry et awaiting_review.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 2 — Étudier une panne

Le processus s’arrête après l’appel LLM mais avant le commit du brouillon : qu’est-ce qui peut se répéter ?

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 3 — Séparer les identifiants

Pourquoi event_id et identifiant d’exécution n8n doivent-ils être distincts ?

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.


# Préparation à l’entretien

Répondre en deux minutes par question, avec un exemple concret et une limite.

## Question 1

Pourquoi ne pas laisser le LLM décider de la transition published ?

## Question 2

Qu’est-ce qu’un checkpoint protège ?


# Challenge

Enregistrer un événement dans une base fichier, fermer le moteur, ouvrir un nouveau moteur puis générer le brouillon. Prouver que le même job est repris.

## Critères de revue

Reproductibilité, absence d’effet non autorisé, explication des limites et preuves du comportement attendu.


# Références — J2

Sources consultées le 12 octobre 2026. Les pages évoluent; les exports ciblent la version n8n indiquée dans Compose.

- [Documentation n8n](https://docs.n8n.io/) — vocabulaire et référence des nœuds.
- [Webhook](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.webhook/) — réception et modes de réponse.
- [HTTP Request](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.httprequest/) — appels au backend et credentials.
- [Gestion des erreurs](https://docs.n8n.io/flow-logic/error-handling/) — séparation du flux nominal et des erreurs techniques.
- [Wait](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.wait/) — variante d’attente, non implémentée ici.
- [sqlite3 Python](https://docs.python.org/3/library/sqlite3.html) — transactions et connexions.

Lire en priorité les éléments liés à l’objectif du jour; les autres liens permettent d’approfondir. Les choix du moteur pédagogique sont explicités dans le cours et ne constituent pas une spécification de n8n.
